In [1]:
# %pip install chromadb

In [2]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np
import chromadb

C:\Users\HP\AppData\Local\Temp\ipykernel_36988\3192942143.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [3]:
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/10201979.pdf')
loader = PyPDFLoader('data/2509.23988v3.pdf')
loader = PyPDFLoader('data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf')
loader = PyPDFLoader('data/1901.08579v1.pdf')
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/2403.08540v2.pdf')
loader = PyPDFLoader('data/2412.11427v2.pdf')
loader = PyPDFLoader('data/2502.04463v4.pdf')
loader = PyPDFLoader('data/2504.01992v1.pdf')
loader = PyPDFLoader('data/2508.14111v2.pdf')
loader = PyPDFLoader('data/2511.01365v1.pdf')
loader = PyPDFLoader('data/Qi_et_al_2026_Towards_trustworthy_agentic_AI_a_comprehensive_survey_of_AcadAI8260.pdf')
loader = PyPDFLoader('data/s10462-026-11571-0.pdf')
loader = PyPDFLoader('data/2001.08361v1.pdf')
loader = PyPDFLoader('data/w35046.pdf')
pages = loader.load()

chroma_client = chromadb.PersistentClient(path="deepseek_vector_db_v2")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AAAAAX+SeasonSans-Regular', '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1800, 0, 2334398724480), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [204, 0, 0, 0, 0, 833, 662, 0, 0, 0, 0, 560, 0, 363, 0, 0, 616, 317, 537, 563, 620, 560, 606, 523, 583, 606, 0, 0, 564, 0, 0, 0, 0, 655, 0, 0, 0, 551, 531, 724, 0, 227, 0, 0, 0, 0, 690, 0, 595, 0, 0, 577, 0, 0, 0, 997, 0, 0, 0, 0, 0, 0, 0, 0, 0, 510, 0, 527, 0, 541, 317, 515, 554, 193, 0, 491, 193, 881, 554, 569, 586, 0, 321, 449, 332, 554, 0, 739, 0, 474]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AAAAAY+SeasonSans-SemiBold', '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(1801, 0, 233439872

### Text Cleaning

In [4]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spaces
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    cleaned_text = cleaned_text.replace("__", "")
    return cleaned_text

In [5]:
for page in pages:
    page.page_content = cleanText(page.page_content)

In [6]:
pages[: 10]

[Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-01T23:59:46+00:00', 'moddate': '2026-04-09T17:08:55-04:00', 'source': 'data/w35046.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1'}, page_content='NBER WORKING PAPER SERIES FORECASTING THE ECONOMIC EFFECTS OF AI Ezra Karger Otto Kuusela Jason Abaluck Kevin A. Bryan Basil Halperin Todd R. Jones Connacher Murphy Philip Trammell Matt Reynolds Dan Mayland Ria Viswanathan Ananaya Mittal Rebecca Ceppas de Castro Josh Rosenberg Philip Tetlock Working Paper 35046 http://www.nber.org/papers/w35046 NATIONAL BUREAU OF ECONOMIC RESEARCH 1050 Massachusetts Avenue Cambridge, MA 02138 April 2026 This research was funded by a grant from Open Philanthropy (now known as Coefficient Giving). Karger and Kuusela are joint first authors. The views expressed in this paper do not necessarily reflect the views of the Federal Reserve Bank of Chicago or the Fed

### Chunking
 - splitting text information into smaller sizes

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 200)

In [8]:
chunks = splitter.split_documents(pages)

In [9]:
chunks[0:10]

[Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-01T23:59:46+00:00', 'moddate': '2026-04-09T17:08:55-04:00', 'source': 'data/w35046.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1'}, page_content='NBER WORKING PAPER SERIES FORECASTING THE ECONOMIC EFFECTS OF AI Ezra Karger Otto Kuusela Jason Abaluck Kevin A. Bryan Basil Halperin Todd R. Jones Connacher Murphy Philip Trammell Matt Reynolds Dan Mayland Ria Viswanathan Ananaya Mittal Rebecca Ceppas de Castro Josh Rosenberg Philip Tetlock Working Paper 35046 http://www.nber.org/papers/w35046 NATIONAL BUREAU OF ECONOMIC RESEARCH 1050'),
 Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-01T23:59:46+00:00', 'moddate': '2026-04-09T17:08:55-04:00', 'source': 'data/w35046.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1'}, page_content='Dan Ma

In [10]:
len(chunks)

635

### Embeddings

In [11]:
def createEmbedding(text):
    response = ollama.embed(
    model = 'nomic-embed-text-v2-moe',
    input = text ,
    )
    return response.embeddings

In [12]:
sample_text = "this is a black car!"
print(createEmbedding(sample_text))

[[0.017525606, -0.055630695, -0.03781086, 0.0278525, 0.014276839, -0.006948967, -0.05646756, -0.018903894, -0.008746914, 0.0019043909, 0.028743535, -0.056393348, 0.07700931, 0.046605684, 0.0015968725, -0.036564734, 0.028226849, -0.0030354303, -0.008055718, 0.061202627, 0.045994826, -0.052638747, -0.03578447, -0.03279134, 0.04578182, -0.0076456754, 0.021769173, 0.021924606, 0.038195003, -0.0159788, 0.0032565072, -0.048310883, -0.0798269, 0.047762796, -0.07094089, 0.004436961, -0.005479182, -0.0046320767, 0.046030004, -0.022270374, 0.023837794, 0.012069714, 0.007284152, 0.06899126, 0.041998155, 0.015125231, -0.0146937845, 0.022237558, -0.019037833, 0.05459687, 0.02681172, 0.015743883, 0.02900423, -0.018440697, 0.041370295, -0.049441222, 0.012728367, -0.02120271, -0.0002063336, -0.0051484145, 0.025235225, 0.024403267, -0.006140415, -0.07623727, -0.06838044, 0.0023302925, 0.033804446, -0.009078112, 0.082351394, 0.009654663, -0.062019415, 0.03309109, -0.048259597, -0.0060315984, 0.008941451

In [13]:
doc_embeddings = []

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding)

In [14]:
len(doc_embeddings)

635

In [15]:
user_query = input('Ask anything about the Document...')


In [16]:
print(user_query)   

how does RAG enhances LLMs?


In [17]:
user_query_embedding  = createEmbedding(user_query)

In [18]:
print(user_query_embedding)

[[0.013303927, -0.074483775, -0.026553012, -0.0029712257, 0.04690742, -0.04753944, -0.0033687274, -0.0063986317, -0.020115923, 0.078084186, -0.08183789, -0.025722947, 0.0042209495, 0.01972109, -0.03656803, -0.06740721, -0.0077790716, 0.03743166, -0.042619463, 0.014035858, 0.0077002323, 0.030331708, -0.064098604, 0.013771542, -0.015219318, 0.058306452, -0.07731662, -0.009083607, 0.039683852, 0.010342302, 0.026210008, 0.018980296, 0.08769567, 0.042929817, 0.0125058815, 0.018191585, -0.04875399, -0.022715207, -0.020320171, -0.030413827, 0.039167717, 0.0014267545, 0.015911138, -0.05861516, -0.012581836, 0.0018110392, -0.12233331, -0.012262249, 0.00641009, 0.003411423, 0.009361511, 0.0067407275, 0.039396543, -0.07188285, 0.06235025, -0.049806524, -0.097283706, -0.063280396, 0.04475386, 0.0072719143, -0.037567195, -0.05624266, 0.018255066, 0.05037871, -0.036172286, 0.059801847, 0.097516984, 0.026020756, -0.03066414, 0.044089608, 0.008717549, 2.8502584e-05, 0.032753732, -0.011217738, -0.04075

In [19]:
above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embedding)
    if score >= 0.2:
        result = (index, score)
        above_zero_point_six.append(result)

In [20]:
above_zero_point_six

[(0, array([[0.21191417]])),
 (1, array([[0.26326303]])),
 (2, array([[0.26381689]])),
 (3, array([[0.22966391]])),
 (4, array([[0.30988668]])),
 (5, array([[0.23006962]])),
 (11, array([[0.2483807]])),
 (12, array([[0.20839036]])),
 (14, array([[0.22261915]])),
 (15, array([[0.21040395]])),
 (16, array([[0.2415789]])),
 (17, array([[0.25341247]])),
 (22, array([[0.26702691]])),
 (23, array([[0.26472122]])),
 (24, array([[0.27088134]])),
 (25, array([[0.27656198]])),
 (26, array([[0.26172605]])),
 (27, array([[0.25875893]])),
 (28, array([[0.24581741]])),
 (29, array([[0.27590752]])),
 (30, array([[0.25080114]])),
 (31, array([[0.21939068]])),
 (32, array([[0.20392852]])),
 (33, array([[0.23727359]])),
 (34, array([[0.24143333]])),
 (35, array([[0.22517484]])),
 (36, array([[0.30670532]])),
 (37, array([[0.28587658]])),
 (38, array([[0.23574371]])),
 (39, array([[0.25372285]])),
 (40, array([[0.25110154]])),
 (41, array([[0.23752696]])),
 (42, array([[0.27922164]])),
 (43, array([[0.26

In [21]:
largest = max(above_zero_point_six, key=lambda x: x[1])
largest

(585, array([[0.45362659]]))

In [22]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

Tyna, Sam Manning, Pamela Mishkin, and Daniel Rock (2024). “GPTs are GPTs: Labor market impact potential of LLMs”. In: Science 384.6702, pp. 1306–1308. Epoch AI (2024). Data on AI Models . url: https://epoch.ai/data/ai models (visited on 11/22/2024). — (2025). AI Capabilities and Benchmarking .O n l i n ed a t a b a s e .A c c e s s e d :2 0 2 6 0 3 3 0 .url: https://epoch.ai/benchmarks. Erdil,


In [23]:
chunks[above_zero_point_six[2][0]].page_content

'Massachusetts Avenue Cambridge, MA 02138 April 2026 This research was funded by a grant from Open Philanthropy (now known as Coefficient Giving). Karger and Kuusela are joint first authors. The views expressed in this paper do not necessarily reflect the views of the Federal Reserve Bank of Chicago or the Federal Reserve System. We thank seminar attendees at the Congressional Budget Office for'

### Evaluations

In [24]:
a = np.array([1,2,3])
b = np.array([1,2,3])

In [25]:
dot_sim = np.dot(a,b)

In [26]:
cos_sim = np.dot(a,b) / (np.linalg.norm(a)* np.linalg.norm(b))

In [27]:
euclidean_dist = np.linalg.norm(a - b)

In [28]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

 
 DOT PRODUCT SIM_SCORE: 14  
 
 COSINE SIM_SCORE: 1.0  
 
 EUCLIDEAN DISTANCE: 0.0  
 



In [29]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [30]:
user_query = input("ask me anything...")

### Creating VectorDB and loading information

In [32]:
collection_research = chroma_client.create_collection(name="deepseek_vector_db_v3")

In [33]:
chunks[1]

Document(metadata={'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-01T23:59:46+00:00', 'moddate': '2026-04-09T17:08:55-04:00', 'source': 'data/w35046.pdf', 'total_pages': 51, 'page': 0, 'page_label': '1'}, page_content='Dan Mayland Ria Viswanathan Ananaya Mittal Rebecca Ceppas de Castro Josh Rosenberg Philip Tetlock Working Paper 35046 http://www.nber.org/papers/w35046 NATIONAL BUREAU OF ECONOMIC RESEARCH 1050 Massachusetts Avenue Cambridge, MA 02138 April 2026 This research was funded by a grant from Open Philanthropy (now known as Coefficient Giving). Karger and Kuusela are joint first authors. The views')

In [34]:
len(doc_embeddings)

635

In [35]:
len(doc_embedding[0])

768

In [36]:
doc_embeddings[234][0]

[-0.016289283,
 0.004140238,
 -0.006554661,
 -0.023209646,
 0.03356647,
 -0.053235207,
 -0.017800739,
 0.06923092,
 0.02592246,
 0.001368883,
 -0.0051922607,
 -0.0047610668,
 -0.031972297,
 0.046728037,
 0.019186832,
 0.04958448,
 0.007341205,
 -0.030265937,
 -0.011239665,
 0.099022634,
 -0.008114665,
 0.011016815,
 -0.074428,
 -0.088473044,
 -0.03487004,
 -0.0044648927,
 -0.03541457,
 -0.001428142,
 0.03144707,
 0.07366312,
 0.033589173,
 0.043648135,
 -0.0057768878,
 0.045413557,
 0.0065094787,
 -0.030968409,
 -0.009819288,
 -0.013740397,
 0.016839454,
 0.060095962,
 0.07289079,
 -0.022056712,
 -0.014598125,
 -0.056356262,
 0.01845667,
 -0.009033503,
 0.021832747,
 0.027861929,
 0.02026109,
 0.03254946,
 0.034191716,
 -0.022404771,
 0.0015723718,
 -0.01246712,
 -0.035815235,
 -0.0932619,
 -0.017484888,
 0.01924826,
 0.051662493,
 -0.010565173,
 0.014884005,
 0.021695115,
 0.0065571615,
 0.049374726,
 0.033991877,
 0.009679859,
 0.056630794,
 0.007682444,
 -0.008991415,
 0.03313698,
 

In [37]:
len(chunks[0].page_content)

388

In [38]:
len(chunks)

635

In [39]:
chunks[1].metadata["creator"]

'LaTeX with hyperref'

In [40]:
for index in range(0,15):
    collection_research.add(
        ids = str(index),
        documents = chunks[index].page_content,
        embeddings=doc_embeddings[index][0],
        metadatas = chunks[index].metadata,
        uris = chunks[index].metadata["creator"]
    )

In [41]:
response = collection_research.query(
     query_texts=["What is Traditional RAG? "], 
     query_embeddings = createEmbedding("What is Traditional RAG? "),
      n_results=4
)

In [ ]:
response["documents"]

[['Massachusetts Avenue Cambridge, MA 02138 April 2026 This research was funded by a grant from Open Philanthropy (now known as Coefficient Giving). Karger and Kuusela are joint first authors. The views expressed in this paper do not necessarily reflect the views of the Federal Reserve Bank of Chicago or the Federal Reserve System. We thank seminar attendees at the Congressional Budget Office for',
  'feedback on this project. We also thank Morgane Bascle, Oskari Luoma, Aashish Reddy, Sam Holton, and Victoria Schmidt for outstanding research assistance. The views expressed herein are those of the authors and do not necessarily reflect the views of the National Bureau of Economic Research. NBER working papers are circulated for discussion and comment purposes. They have not been peer reviewed',
  'authors and do not necessarily reflect the views of the National Bureau of Economic Research. NBER working papers are circulated for discussion and comment purposes. They have not been peer re

In [ ]:
for index in range(0,len(chunks)):
    collection_research.add(
        ids = str(index),
        documents = chunks[index].page_content,
        embeddings= doc_embeddings[index][0],
        metadatas = chunks[index].metadata ,
        uris = chunks[index].metadata["creator"]
    )

In [ ]:

collection_research.get(ids=["0"]) # by IDs

{'ids': ['0'],
 'embeddings': None,
 'documents': ['NBER WORKING PAPER SERIES FORECASTING THE ECONOMIC EFFECTS OF AI Ezra Karger Otto Kuusela Jason Abaluck Kevin A. Bryan Basil Halperin Todd R. Jones Connacher Murphy Philip Trammell Matt Reynolds Dan Mayland Ria Viswanathan Ananaya Mittal Rebecca Ceppas de Castro Josh Rosenberg Philip Tetlock Working Paper 35046 http://www.nber.org/papers/w35046 NATIONAL BUREAU OF ECONOMIC RESEARCH 1050'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'page_label': '1',
   'source': 'data/w35046.pdf',
   'moddate': '2026-04-09T17:08:55-04:00',
   'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext',
   'page': 0,
   'creator': 'LaTeX with hyperref',
   'total_pages': 51,
   'creationdate': '2026-04-01T23:59:46+00:00'}]}

In [ ]:

# query 
response = collection_research.query(
     query_texts=["What is Traditional RAG? "], 
     query_embeddings = createEmbedding("What is Traditional RAG? "),
      n_results=7
)

In [ ]:

response.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [ ]:
print(response["embeddings"])

None


In [ ]:
print(response["data"])

None


In [ ]:

print(response["uris"])

None


In [ ]:
response["metadatas"]

[[{'moddate': '2026-04-09T17:08:55-04:00',
   'source': 'data/w35046.pdf',
   'page_label': '9',
   'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext',
   'total_pages': 51,
   'creator': 'LaTeX with hyperref',
   'page': 8,
   'creationdate': '2026-04-01T23:59:46+00:00'},
  {'creator': 'LaTeX with hyperref',
   'source': 'data/w35046.pdf',
   'total_pages': 51,
   'page_label': '23',
   'page': 22,
   'moddate': '2026-04-09T17:08:55-04:00',
   'creationdate': '2026-04-01T23:59:46+00:00',
   'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext'},
  {'moddate': '2026-04-09T17:08:55-04:00',
   'page_label': '23',
   'producer': 'macOS Version 15.7.4 (Build 24G517) Quartz PDFContext',
   'creator': 'LaTeX with hyperref',
   'page': 22,
   'total_pages': 51,
   'source': 'data/w35046.pdf',
   'creationdate': '2026-04-01T23:59:46+00:00'},
  {'source': 'data/w35046.pdf',
   'page_label': '15',
   'creationdate': '2026-04-01T23:59:46+00:00',
   'total_pages': 51

In [ ]:
response["distances"]

[[1.4901981353759766,
  1.5075608491897583,
  1.525256872177124,
  1.5302822589874268,
  1.5428987741470337,
  1.5448551177978516,
  1.5610836744308472]]

In [ ]:
response["documents"]

[['to taxation and redistribution at a scale with no peacetime precedent. Comunale and Manera ( 2024) and the OECD (OECD, 2023)b o t hc a t a l o ge x i s t i n gr e g u l a t o r ya p p r o a c h e sa n ds t r e s st h a to p t i m a li n t e r v e n t i o n sv a r y 3AGI stands for Artiﬁcial General Intelligence and refers to a world where AI systems can perform tasks and bundles of tasks at a',
  'and personal computers to argue that multi decade lags routinely separate a general purpose technology’s arrival from its measurable productivity impact. Geopolitical, structural, and demographic headwinds—including trade wars, climate change, an aging population, and declining immigration—that could o!set AI driven gains were also cited, as were constraints on energy, chips, and data center',
  'reason why transformative technology would translate into only modest economic growth was uneven and time lagged di!usion, with economists drawing on analogies to electriﬁcation, automobiles, and 

In [ ]:

response["documents"][0][0]

'to taxation and redistribution at a scale with no peacetime precedent. Comunale and Manera ( 2024) and the OECD (OECD, 2023)b o t hc a t a l o ge x i s t i n gr e g u l a t o r ya p p r o a c h e sa n ds t r e s st h a to p t i m a li n t e r v e n t i o n sv a r y 3AGI stands for Artiﬁcial General Intelligence and refers to a world where AI systems can perform tasks and bundles of tasks at a'

In [ ]:
response["documents"][0][-1]

'Kits, Remi Sebastian (Feb. 2, 2026). Weightipy. A modern, high performance RIM (iterative raking) weighting library for survey data .V e r s i o n0 . 4 . 2 .D o c u m e n t a t i o na v a i l a b l ea t https://kaitumisuuringute keskus.github.io/Weightipy/. url: https://github. com/kaitumisuuringute keskus/Weightipy (visited on 03/20/2026). Korinek, Anton (2024). Economic Consequences of Frontier'